<a href="https://colab.research.google.com/github/caihualiang5-svg/protien-Nampt/blob/mutation/1_ESM_and_Conservation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 — ESM zero-shot scores and sequence conservation

**Input:** one WT amino-acid sequence.  
**Output:** `all_mutations_esm_conservation.csv` (one row per one-step
amino-acid substitution).

Smoke mode uses the fixed 50-aa fixture and deterministic fake scores;
it never downloads ESM weights or calls an MSA service. Production mode
uses exactly one ESM-2 checkpoint and one ESM-1v checkpoint. Every ESM
percentile is calculated over the complete `19 × L` mutation table.


In [ ]:
from pathlib import Path

TEST_MODE = False
PROTEIN_ID = "FAKE50" if TEST_MODE else "NAMPT_WT"
WT_SEQUENCE = (
    "ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL"
    if TEST_MODE else "MQPNIILLTDSYKLSHYKQYPAGTSQIYSYFESRGGEFEGVTFFGLQYLLKEYLEGQVVTQEKIDRADKIYAAHFGTEKLFNKAGWEYILHTHNGHLPIRIKAVAEGTVIPTHNVMLTIENTDPNCFWLTNFLETLLLQLWYPCTVATISREVKTLITKYLEETGDPSTIDFKLHDFGFRGVSSVQSAGIGGAAHLVNFMGTDTVAALTFIQEYYAPFPVGEGLGMGFPMFGFSIPAAEHSTITSWGRDNETDAYQNMLQQYPEGLVAVVSDSYDIYNACEKIWGEVLKDNILQRNGTLVVRPDSGEPKDVVLKCTQILGEKIGYSINEKGYKVLNPKIRIIQGDGVNYESIGEILEHLKKHGWSADNVAFGMGGALLQKVHRDTQKFAFKCSCATVNGEDRDVYKDPATDHGKKSKRGRLKLVKENEMYITKAINEDGEDILQTVFENGKILREIDFQGVKENNLK"
)
ESM2_MODEL_ID = "esm2_t33_650M_UR50D"
ESM1V_MODEL_ID = "esm1v_t33_650M_UR90S_1"
ESM_BATCH_SIZE = 4
DRIVE_DIR = Path("/content/drive/MyDrive/nampt_zero_shot")
LOCAL_OUTPUT_DIR = Path("/content/nampt_zero_shot")


In [ ]:
import sys
import subprocess

def module1_install_requirements(test_mode: bool) -> list[str]:
    if test_mode:
        return []
    return [
        "fair-esm==2.0.0",
        "requests>=2.32,<3",
        "colabfold==1.6.2",
    ]


IN_COLAB = "google.colab" in sys.modules
INSTALL_REQUIREMENTS = module1_install_requirements(TEST_MODE)
if IN_COLAB and INSTALL_REQUIREMENTS:
    print("Installing production-only ESM/MMseqs2 dependencies...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        *INSTALL_REQUIREMENTS,
    ])
elif TEST_MODE:
    print("Smoke mode: using Colab's built-in NumPy/Pandas; no model packages installed.")
else:
    print("Not running in Colab; dependency installation skipped.")


Installing production-only ESM/MMseqs2 dependencies...


In [ ]:
UPLOADED_INPUTS = {}
if IN_COLAB:
    from google.colab import files
    UPLOADED_INPUTS = files.upload()
else:
    print("Upload widget is available only in Colab; using configured inputs.")


In [ ]:
DRIVE_MOUNTED = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
else:
    print("Drive mount is available only in Colab.")


Mounted at /content/drive


In [ ]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

AA20 = "ACDEFGHIKLMNPQRSTVWY"


def validate_wt_sequence(sequence: str) -> str:
    cleaned = "".join(sequence.split()).upper()
    invalid = sorted(set(cleaned) - set(AA20))
    if not cleaned or invalid:
        raise ValueError(f"Invalid WT sequence residues: {invalid}")
    return cleaned


def sequence_sha256(sequence: str) -> str:
    return hashlib.sha256(sequence.encode("ascii")).hexdigest()


def enumerate_single_mutants(sequence: str, protein_id: str) -> pd.DataFrame:
    sequence = validate_wt_sequence(sequence)
    rows = []
    for position, wt_aa in enumerate(sequence, start=1):
        for mut_aa in AA20:
            if mut_aa == wt_aa:
                continue
            mutant = sequence[: position - 1] + mut_aa + sequence[position:]
            rows.append({
                "protein_id": protein_id,
                "sequence_hash": sequence_sha256(sequence),
                "mutation_id": f"{wt_aa}{position}{mut_aa}",
                "position": position,
                "wt_aa": wt_aa,
                "mut_aa": mut_aa,
                "mutant_sequence": mutant,
                "row_status": "success",
                "error_message": "",
            })
    result = pd.DataFrame(rows)
    expected = 19 * len(sequence)
    if len(result) != expected or result["mutation_id"].nunique() != expected:
        raise AssertionError(f"Expected {expected} unique mutants")
    if not result.groupby("position").size().eq(19).all():
        raise AssertionError("Each WT position must have exactly 19 substitutions")
    return result


def add_global_percentiles(
    frame: pd.DataFrame, score_columns: list[str]
) -> pd.DataFrame:
    result = frame.copy()
    reference_n = len(result)
    for score_column in score_columns:
        if score_column not in result:
            raise KeyError(f"Missing score column: {score_column}")
        model_prefix = score_column.removesuffix("_score")
        result[f"{model_prefix}_percentile"] = result[score_column].rank(
            method="average", pct=True, ascending=True
        )
        result[f"{model_prefix}_percentile_reference_n"] = reference_n
    if reference_n != 19 * result["position"].nunique():
        raise AssertionError(
            "ESM percentile reference is not the complete 19 × L set"
        )
    return result


def deterministic_unit_value(label: str) -> float:
    digest = hashlib.sha256(label.encode("utf-8")).digest()
    integer = int.from_bytes(digest[:8], byteorder="big", signed=False)
    return integer / float(2**64 - 1)


def add_deterministic_smoke_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    result["esm2_score"] = [
        6.0 * deterministic_unit_value(f"esm2:{m}") - 4.0
        for m in result["mutation_id"]
    ]
    result["esm1v_score"] = [
        6.0 * deterministic_unit_value(f"esm1v:{m}") - 4.0
        for m in result["mutation_id"]
    ]
    result["msa_conservation"] = [
        deterministic_unit_value(f"msa:{p}") for p in result["position"]
    ]
    result["msa_entropy"] = 1.0 - result["msa_conservation"]
    result["mutant_msa_frequency"] = [
        deterministic_unit_value(f"msa:{p}:{aa}")
        for p, aa in zip(result["position"], result["mut_aa"])
    ]
    result["msa_depth"] = 128
    result["msa_source"] = "deterministic_smoke_fixture"
    return add_global_percentiles(
        result, ["esm2_score", "esm1v_score"]
    )


def parse_a3m_conservation(
    a3m_text: str, wt_sequence: str
) -> pd.DataFrame:
    import string

    wt_sequence = validate_wt_sequence(wt_sequence)
    records = []
    header = None
    sequence_parts = []
    for raw_line in a3m_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                records.append((header, "".join(sequence_parts)))
            header = line[1:].strip() or f"sequence_{len(records) + 1}"
            sequence_parts = []
        else:
            if header is None:
                raise ValueError("A3M sequence appeared before the first header")
            sequence_parts.append(line)
    if header is not None:
        records.append((header, "".join(sequence_parts)))
    if not records:
        raise ValueError("A3M contains no sequences")
    delete = dict.fromkeys(map(ord, string.ascii_lowercase + ".*"))
    aligned = [sequence.translate(delete).upper() for _, sequence in records]
    query_ungapped = aligned[0].replace("-", "")
    if query_ungapped != wt_sequence:
        raise ValueError("The first A3M sequence does not map exactly to the WT")
    if any(len(sequence) != len(aligned[0]) for sequence in aligned):
        raise ValueError("A3M rows have inconsistent aligned lengths")
    query_columns = [
        index for index, residue in enumerate(aligned[0]) if residue != "-"
    ]
    if len(query_columns) != len(wt_sequence):
        raise ValueError("A3M query-to-WT column mapping is incomplete")
    msa_depth = len(aligned)
    effective_sequences = len(set(aligned))
    rows = []
    for position, column in enumerate(query_columns, start=1):
        column_values = [sequence[column] for sequence in aligned]
        valid = [residue for residue in column_values if residue in AA20]
        counts = {aa: valid.count(aa) for aa in AA20}
        denominator = len(valid)
        frequencies = {
            aa: (counts[aa] / denominator if denominator else 0.0)
            for aa in AA20
        }
        probabilities = [value for value in frequencies.values() if value > 0]
        entropy = -sum(p * np.log(p) for p in probabilities) / np.log(20.0)
        consensus = max(AA20, key=lambda aa: (counts[aa], -AA20.index(aa)))
        row = {
            "position": position,
            "msa_query_aa": wt_sequence[position - 1],
            "msa_consensus_aa": consensus,
            "msa_entropy": float(entropy),
            "msa_conservation": float(1.0 - entropy),
            "msa_gap_fraction": column_values.count("-") / msa_depth,
            "msa_depth": msa_depth,
            "msa_effective_sequences": effective_sequences,
        }
        row.update({f"msa_freq_{aa}": frequencies[aa] for aa in AA20})
        rows.append(row)
    return pd.DataFrame(rows)


def validate_module1_output(
    frame: pd.DataFrame, sequence: str
) -> None:
    sequence = validate_wt_sequence(sequence)
    required = {
        "protein_id", "sequence_hash", "mutation_id", "position",
        "wt_aa", "mut_aa", "mutant_sequence", "esm2_score",
        "esm1v_score", "esm2_percentile", "esm1v_percentile",
        "esm2_percentile_reference_n", "esm1v_percentile_reference_n",
        "msa_conservation", "msa_entropy", "row_status",
        "error_message",
    }
    missing = sorted(required - set(frame.columns))
    if missing:
        raise ValueError(f"Module 1 is missing required columns: {missing}")
    expected = 19 * len(sequence)
    if len(frame) != expected or frame["mutation_id"].nunique() != expected:
        raise ValueError(f"Module 1 must contain {expected} unique mutations")
    if frame["position"].nunique() != len(sequence):
        raise ValueError("Module 1 does not cover every WT position")
    if not frame.groupby("position").size().eq(19).all():
        raise ValueError("Every position must contain 19 substitutions")
    expected_hash = sequence_sha256(sequence)
    if not frame["sequence_hash"].eq(expected_hash).all():
        raise ValueError("Module 1 sequence_hash does not match the WT")
    if frame[["esm2_score", "esm1v_score", "esm2_percentile", "esm1v_percentile"]].isna().any().any():
        raise ValueError("ESM scores and percentiles must be complete")
    if not frame["row_status"].eq("success").all():
        raise ValueError("Validated module-1 rows must have row_status='success'")
    for model_prefix in ["esm2", "esm1v"]:
        reference_column = f"{model_prefix}_percentile_reference_n"
        if not frame[reference_column].eq(expected).all():
            raise ValueError(
                f"{reference_column} must equal the global 19 × L size {expected}"
            )
        recomputed = frame[f"{model_prefix}_score"].rank(
            method="average", pct=True, ascending=True
        )
        stored = frame[f"{model_prefix}_percentile"]
        if not np.allclose(recomputed, stored, rtol=0, atol=1e-12):
            raise ValueError(
                f"{model_prefix} percentiles were not computed globally"
            )
    for row in frame.itertuples():
        if row.wt_aa != sequence[row.position - 1]:
            raise ValueError(f"WT residue mismatch at {row.mutation_id}")
        if len(row.mutant_sequence) != len(sequence):
            raise ValueError(
                f"Mutant sequence length mismatch at {row.mutation_id}: "
                f"{len(row.mutant_sequence)} != {len(sequence)}"
            )
        differences = sum(
            aa_wt != aa_mut
            for aa_wt, aa_mut in zip(sequence, row.mutant_sequence)
        )
        if differences != 1 or row.mutant_sequence[row.position - 1] != row.mut_aa:
            raise ValueError(f"Invalid single mutant: {row.mutation_id}")


In [ ]:
WT50 = "ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWYACDEFGHIKL"
smoke_output_dir = Path(globals().get("SMOKE_OUTPUT_DIR", "/content"))
smoke_output_dir.mkdir(parents=True, exist_ok=True)
smoke_frame = enumerate_single_mutants(WT50, "FAKE50")
smoke_frame = add_deterministic_smoke_features(smoke_frame)
assert len(WT50) == 50
assert len(smoke_frame) == 950
assert smoke_frame["mutation_id"].nunique() == 950
assert smoke_frame["esm2_percentile_reference_n"].eq(950).all()
assert smoke_frame["esm1v_percentile_reference_n"].eq(950).all()
smoke_path = smoke_output_dir / "all_mutations_esm_conservation_smoke.csv"
smoke_frame.to_csv(smoke_path, index=False)
print(f"Smoke output: {smoke_path} ({len(smoke_frame)} rows)")


Smoke output: /content/all_mutations_esm_conservation_smoke.csv (950 rows)


In [ ]:
def load_configured_esm(model_id: str):
    import esm

    loaders = {
        "esm2_t33_650M_UR50D": esm.pretrained.esm2_t33_650M_UR50D,
        "esm1v_t33_650M_UR90S_1": esm.pretrained.esm1v_t33_650M_UR90S_1,
    }
    if model_id not in loaders:
        raise ValueError(f"Unapproved ESM checkpoint: {model_id}")
    model, alphabet = loaders[model_id]()
    model.eval()
    return model, alphabet


def score_masked_marginals(
    sequence,
    model,
    alphabet,
    batch_size=1,
    checkpoint_path=None,
    model_name="esm",
):
    import gc
    import torch

    sequence = validate_wt_sequence(sequence)
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    checkpoint_path = Path(checkpoint_path) if checkpoint_path else None
    completed = pd.DataFrame()
    completed_positions = set()
    if checkpoint_path is not None and checkpoint_path.exists():
        completed = pd.read_csv(checkpoint_path)
        completed_positions = set(completed["position"].astype(int))
    batch_converter = alphabet.get_batch_converter()
    _, _, base_tokens = batch_converter([("WT", sequence)])
    device = next(model.parameters()).device
    base_tokens = base_tokens.to(device)
    new_rows = []
    pending = [
        position for position in range(1, len(sequence) + 1)
        if position not in completed_positions
    ]
    try:
        for start in range(0, len(pending), batch_size):
            positions = pending[start:start + batch_size]
            batch_tokens = base_tokens.repeat(len(positions), 1)
            for row_index, position in enumerate(positions):
                batch_tokens[row_index, position] = alphabet.mask_idx
            with torch.no_grad():
                autocast_enabled = device.type == "cuda"
                with torch.autocast(
                    device_type=device.type,
                    dtype=torch.float16,
                    enabled=autocast_enabled,
                ):
                    logits = model(batch_tokens)["logits"]
                log_probs = torch.log_softmax(logits.float(), dim=-1)
            for row_index, position in enumerate(positions):
                wt_aa = sequence[position - 1]
                wt_log_prob = log_probs[
                    row_index, position, alphabet.get_idx(wt_aa)
                ].item()
                for mut_aa in AA20:
                    if mut_aa == wt_aa:
                        continue
                    mutant_log_prob = log_probs[
                        row_index, position, alphabet.get_idx(mut_aa)
                    ].item()
                    new_rows.append({
                        "position": position,
                        "wt_aa": wt_aa,
                        "mut_aa": mut_aa,
                        "mutation_id": f"{wt_aa}{position}{mut_aa}",
                        f"{model_name}_score": mutant_log_prob - wt_log_prob,
                    })
            if checkpoint_path is not None:
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                pd.concat(
                    [completed, pd.DataFrame(new_rows)], ignore_index=True
                ).drop_duplicates("mutation_id", keep="last").to_csv(
                    checkpoint_path, index=False
                )
            del batch_tokens, logits, log_probs
            if device.type == "cuda":
                torch.cuda.empty_cache()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            if checkpoint_path is not None:
                checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
                pd.concat(
                    [completed, pd.DataFrame(new_rows)], ignore_index=True
                ).drop_duplicates("mutation_id", keep="last").to_csv(
                    checkpoint_path, index=False
                )
            raise RuntimeError(
                "CUDA OOM. ESM checkpoint saved. Manually restart a clean "
                "Colab runtime and rerun to resume. Automatic reconnect is disabled."
            ) from exc
        raise
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    result = pd.concat(
        [completed, pd.DataFrame(new_rows)], ignore_index=True
    ).drop_duplicates("mutation_id", keep="last")
    expected = 19 * len(sequence)
    if len(result) != expected:
        raise RuntimeError(
            f"Incomplete {model_name} score table: {len(result)}/{expected}"
        )
    return result.sort_values(["position", "mut_aa"]).reset_index(drop=True)


def run_one_esm_checkpoint(sequence, model_id, model_name, checkpoint_path):
    import gc
    import torch

    model, alphabet = load_configured_esm(model_id)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError(
            "A Colab GPU runtime is required for the 650M ESM checkpoints."
        )
    model = model.to(device)
    try:
        return score_masked_marginals(
            sequence,
            model,
            alphabet,
            batch_size=ESM_BATCH_SIZE,
            checkpoint_path=checkpoint_path,
            model_name=model_name,
        )
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


if TEST_MODE:
    print("Smoke mode: production ESM checkpoints are not loaded.")
else:
    WT_SEQUENCE = validate_wt_sequence(WT_SEQUENCE)
    esm_checkpoint_dir = DRIVE_DIR / "module1" / "checkpoints"
    esm_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    esm2_scores = run_one_esm_checkpoint(
        WT_SEQUENCE,
        ESM2_MODEL_ID,
        "esm2",
        esm_checkpoint_dir / "esm2_masked_marginals.csv",
    )
    esm1v_scores = run_one_esm_checkpoint(
        WT_SEQUENCE,
        ESM1V_MODEL_ID,
        "esm1v",
        esm_checkpoint_dir / "esm1v_masked_marginals.csv",
    )
    module1_frame = enumerate_single_mutants(WT_SEQUENCE, PROTEIN_ID)
    module1_frame = module1_frame.merge(
        esm2_scores[["mutation_id", "esm2_score"]],
        on="mutation_id", how="left", validate="one_to_one",
    ).merge(
        esm1v_scores[["mutation_id", "esm1v_score"]],
        on="mutation_id", how="left", validate="one_to_one",
    )
    module1_frame = add_global_percentiles(
        module1_frame, ["esm2_score", "esm1v_score"]
    )


Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm1v_t33_650M_UR90S_1.pt" to /root/.cache/torch/hub/checkpoints/esm1v_t33_650M_UR90S_1.pt


/usr/local/lib/python3.13/dist-packages/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


In [ ]:
from pathlib import Path


def fetch_mmseqs2_a3m(sequence, prefix, runner=None):
    sequence = validate_wt_sequence(sequence)
    if runner is None:
        from colabfold.colabfold import run_mmseqs2
        runner = run_mmseqs2
    a3m_lines = runner(
        [sequence],
        str(prefix),
        use_env=True,
        use_filter=True,
        use_templates=False,
        use_pairing=False,
        user_agent="nampt-zero-shot-colab/1.0",
    )
    if len(a3m_lines) != 1 or not a3m_lines[0].strip():
        raise RuntimeError("MMseqs2 API returned no usable A3M alignment")
    return a3m_lines[0]


def generate_mmseqs2_a3m(sequence, protein_id, output_dir):
    sequence = validate_wt_sequence(sequence)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fasta_path = output_dir / f"{protein_id}.fasta"
    fasta_path.write_text(f">{protein_id}\n{sequence}\n", encoding="utf-8")
    stderr_path = output_dir / "mmseqs2_stderr.log"
    try:
        a3m_text = fetch_mmseqs2_a3m(
            sequence, output_dir / f"{protein_id}_mmseqs2"
        )
    except Exception as exc:
        error_text = f"{type(exc).__name__}: {exc}"
        stderr_path.write_text(error_text + "\n", encoding="utf-8")
        raise RuntimeError(
            f"MMseqs2 API query failed: {error_text}"
        ) from exc
    a3m_path = output_dir / f"{protein_id}.a3m"
    a3m_path.write_text(a3m_text, encoding="utf-8")
    (output_dir / "mmseqs2_stdout.log").write_text(
        f"Saved lightweight ColabFold MMseqs2 API result to {a3m_path}\n",
        encoding="utf-8",
    )
    stderr_path.write_text("", encoding="utf-8")
    return a3m_path


A3M_FILENAME = f"{PROTEIN_ID}.a3m"
if TEST_MODE:
    print("Smoke mode: MMseqs2/A3M query is skipped.")
else:
    msa_dir = DRIVE_DIR / "module1" / "msa"
    a3m_path = msa_dir / A3M_FILENAME
    if not a3m_path.exists():
        a3m_path = generate_mmseqs2_a3m(
            WT_SEQUENCE, PROTEIN_ID, msa_dir
        )
    conservation = parse_a3m_conservation(
        a3m_path.read_text(encoding="utf-8"), WT_SEQUENCE
    )
    module1_frame = module1_frame.merge(
        conservation, on="position", how="left", validate="many_to_one"
    )
    frequency_columns = {aa: f"msa_freq_{aa}" for aa in AA20}
    module1_frame["mutant_msa_frequency"] = [
        module1_frame.at[index, frequency_columns[mut_aa]]
        for index, mut_aa in zip(
            module1_frame.index, module1_frame["mut_aa"]
        )
    ]
    module1_frame["msa_source"] = "mmseqs2_colabfold_a3m"
    validate_module1_output(module1_frame, WT_SEQUENCE)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:02 remaining: 00:00]


In [ ]:
output_frame = smoke_frame if TEST_MODE else globals().get("module1_frame")
if output_frame is None:
    raise RuntimeError("No module-1 output frame is available")
expected_rows = 19 * output_frame["position"].nunique()
if len(output_frame) != expected_rows:
    raise AssertionError("Output is not the complete 19 × L mutation table")
validate_module1_output(
    output_frame, WT50 if TEST_MODE else WT_SEQUENCE
)
local_dir = Path(globals().get("LOCAL_OUTPUT_DIR", "/content/nampt_zero_shot"))
local_dir.mkdir(parents=True, exist_ok=True)
output_name = (
    "all_mutations_esm_conservation_smoke.csv"
    if TEST_MODE else "all_mutations_esm_conservation.csv"
)
local_path = local_dir / output_name
output_frame.to_csv(local_path, index=False)
print(f"Saved local CSV: {local_path}")
if globals().get("DRIVE_MOUNTED", False):
    drive_path = DRIVE_DIR / output_name
    output_frame.to_csv(drive_path, index=False)
    print(f"Saved Drive CSV: {drive_path}")


Saved local CSV: /content/nampt_zero_shot/all_mutations_esm_conservation.csv
Saved Drive CSV: /content/drive/MyDrive/nampt_zero_shot/all_mutations_esm_conservation.csv


In [ ]:
from pathlib import Path
import pandas as pd

output_path = Path(
    "/content/drive/MyDrive/nampt_zero_shot/"
    "all_mutations_esm_conservation.csv"
)

df = pd.read_csv(output_path)
L = df["position"].nunique()

print("序列长度 L:", L)
print("实际突变行数:", len(df))
print("理论突变行数 19×L:", 19 * L)
print("唯一突变数:", df["mutation_id"].nunique())
print("ESM-2 缺失值:", df["esm2_score"].isna().sum())
print("ESM-1v 缺失值:", df["esm1v_score"].isna().sum())
print("MSA 深度范围:", df["msa_depth"].min(), "—", df["msa_depth"].max())
print("MSA 来源:", df["msa_source"].value_counts().to_dict())
print("行状态:", df["row_status"].value_counts().to_dict())

序列长度 L: 467
实际突变行数: 8873
理论突变行数 19×L: 8873
唯一突变数: 8873
ESM-2 缺失值: 0
ESM-1v 缺失值: 0
MSA 深度范围: 5558 — 5558
MSA 来源: {'mmseqs2_colabfold_a3m': 8873}
行状态: {'success': 8873}
